In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_absolute_error , mean_absolute_percentage_error , mean_squared_error,r2_score


In [3]:
df = pd.read_csv("Air Traffic Data Cor Updated.csv",parse_dates=['Date'],index_col='Date')
df.head()

,domestic passengers,international passenegrs,domestic freight(in tonne),international freight(in tonne),GDP (in dollars),Jet Fuel Price per Gallon,Inflation Rate,Unemployement Rate
Date,,,,,,,,
2009-01-01,3288004,885435,20832,11675,1.341888e+12,71.75,10.88,7.66
2009-01-02,3293220,757168,18645,12482,1.341888e+12,61.97,10.88,7.66
2009-01-03,3122400,848046,23046,15359,1.341888e+12,65.01,10.88,7.66
2009-01-04,3266686,861715,21623,14512,1.341888e+12,68.55,10.88,7.66
2009-01-05,3883887,898410,19534,14586,1.341888e+12,72.22,10.88,7.66


In [4]:
features = df.copy()

Scaling of Data using MinMaxScaler

In [5]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(features)

Setting Time Series Data into LSTM

In [6]:
def create_sequences_multivariate(data, seq_length=12):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length, :])     # past 12 months, all variables
        y.append(data[i+seq_length, :])       # next step, all variables
    return np.array(X), np.array(y)

SEQ_LEN = 12
X, y = create_sequences_multivariate(scaled_data, SEQ_LEN)

print("X shape:", X.shape)  # (samples, 12, num_features)
print("y shape:", y.shape)  # (samples, num_features)

X shape: (180, 12, 8)
y shape: (180, 8)


Train - Test Split

In [7]:
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

In [8]:
n_features = X.shape[2] # no of columns in dataset

Model Building

In [9]:
model_lstm = Sequential([
    LSTM(64, activation='tanh', return_sequences=True, input_shape=(SEQ_LEN, n_features)),
    Dropout(0.2),
    LSTM(32, activation='tanh'),
    Dense(n_features)  # predict all features (multi-output)
])

C:\Users\abhij\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
model_lstm.compile(optimizer='adam', loss='mse')
history = model_lstm.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.1, verbose=1)

Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 153ms/step - loss: 0.2783 - val_loss: 0.1218
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.1114 - val_loss: 0.0610
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.0485 - val_loss: 0.0574
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0435 - val_loss: 0.0423
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.0338 - val_loss: 0.0365
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.0304 - val_loss: 0.0407
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0285 - val_loss: 0.0439
Epoch 8/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.0250 - val_loss: 0.0423
Epoch 9/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0235 - val_loss: 0.0411
Epoch 10/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.0202 - val_loss: 0.0408
Epoch 11/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0179 - val_loss: 0.0338
Epoch 12/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.0185 - val_loss: 0.0301


Evaluation of Model

In [11]:
y_pred_lstm = model_lstm.predict(X_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 357ms/step


In [12]:
y_test_inv = scaler.inverse_transform(y_test)
y_pred_lstm_inv = scaler.inverse_transform(y_pred_lstm)

In [18]:
mae = mean_absolute_error(y_test_inv[:,2], y_pred_lstm_inv[:,2])
rmse = np.sqrt(mean_squared_error(y_test_inv[:,2], y_pred_lstm_inv[:,2]))
mape = mean_absolute_percentage_error(y_test_inv[:,2] , y_pred_lstm_inv[:,2])*100
r2 = r2_score(y_test_inv[:,2] , y_pred_lstm_inv[:,2])
print(f"LSTM -> Domestic Freight: MAE={mae:.2f}\n RMSE={rmse:.2f}\n MAPE={mape:.2f}\n r2 score={r2}")

LSTM -> Domestic Freight: MAE=31705.03
 RMSE=32428.01
 MAPE=51.15
 r2 score=-38.80224550933854


In [19]:
mae = mean_absolute_error(y_test_inv[:,0], y_pred_lstm_inv[:,0])
rmse = np.sqrt(mean_squared_error(y_test_inv[:,0], y_pred_lstm_inv[:,0]))
mape = mean_absolute_percentage_error(y_test_inv[:,0] , y_pred_lstm_inv[:,0])*100
r2 = r2_score(y_test_inv[:,0] , y_pred_lstm_inv[:,0])
print(f"LSTM -> Domestic Passengers: MAE={mae:.2f}\n RMSE={rmse:.2f}\n MAPE={mape:.2f}\n r2 score={r2}")

LSTM -> Domestic Passengers: MAE=6245497.93
 RMSE=6587013.15
 MAPE=49.73
 r2 score=-13.787348964986377
